<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200"/>
</p></center>

<center><font size=10>Artificial Intelligence and Machine Learning</center></font>
<center><font size=6>Deploying GenAI solutions</center></font>

<center><p float="center">
  <img src="https://images.pexels.com/photos/28238364/pexels-photo-28238364.jpeg" alt="hotel-qna" width="640"/>
</p></center>
<center><font size=6>AuroraStays QnA</center></font>

# Problem Statement

## Business Context

**AuroraStay Hotels** is a mid-sized hotel group operating **18 business and leisure properties** across major cities in the USA, adding **3 to 4 new properties every year**.

Hotel staff handle around **1,200 guest queries per week** about:

- Room types
- Check-in/check-out
- Cancellation policies
- Refunds
- Breakfast
- Parking
- Pets
- Amenities
- Additional charges

Simple queries can be answered using existing FAQs. However, around **40% of queries are complex** and require information from multiple documents, such as:

- Booking terms
- Cancellation policies
- Room information
- Guest guidelines

Manually combining this information takes **8 to 12 minutes per query** and results in a **15% to 20% error rate** because answers may be incomplete or incorrect.

As the hotel group grows, the number of properties and property-specific policies also increases. With **45% annual staff turnover**, institutional knowledge is constantly lost.

## Key Risks

- **Faithfulness:** Avoid unsupported information.
- **Retrieval Completeness:** Retrieve all relevant information required to answer the query.
- **Retrieval Precision:** Retrieve relevant source chunks while minimizing irrelevant information.
- **Answer Relevancy:** Directly answer the guest's question.

A **RAG system** can retrieve relevant information from these documents and help staff provide **accurate, source-grounded answers**.

## Objective

The objective is to build a **POC for a user interface** for AuroraStay Hotels where hotel staff can enter natural-language questions and receive answers based on the hotel's documents.

The application should:

- Provide a simple UI where staff can enter their questions.
- Retrieve the most relevant information from the hotel's document collection.
- Generate an answer grounded **only in the retrieved information**.
- Display the supporting source information along with the answer.
- **Reduce the number of queries that require manual lookup or escalation**, rather than focusing on reducing the time spent handling each query.

Since this is a **POC**, the application will initially be **internal-facing**, allowing the hotel team to test it extensively, ensure that user queries are answered consistently, and validate the **accuracy and reliability** of the system before deploying it for broader use.

> **Note:** The objective at this stage is **not to reduce query-handling time**. The primary objective is to determine whether the system can reliably answer a meaningful proportion of queries without requiring manual effort.

## Success Criteria

| **Business Objective** | **Technical Metric** | **Threshold** |
|---|---|---:|
| Avoid unsupported information | Faithfulness | >= 0.85 |
| Retrieve all relevant information | Contextual Recall | >= 0.80 |
| Retrieve relevant chunks | Contextual Precision | >= 0.75 |
| Answer the question asked | Answer Relevancy | >= 0.75 |
| Reduce the number of queries requiring manual searches for policy information | Queries Confidently Answered | >= 50% |

**Composite Score = 0.35 × Faithfulness + 0.25 × Contextual Recall + 0.20 × Contextual Precision + 0.20 × Answer Relevancy**

The weights for the composite score are **informed by past evaluation results and the relative importance of each metric to the business objectives**. Faithfulness is given the highest weight because avoiding unsupported or misleading information is the most critical requirement in this use case.

The pipeline routes every generated answer through a **scoring stage**. Based on the composite score, the answer is routed to one of two actions:

1. **Auto Answer (score >= 75%)**  
   The generated answer and its supporting source chunks are considered reliable enough to present directly to the representative for immediate use with the customer.

2. **Manual Lookup (score < 75%)**  
   The system output is not considered reliable enough to act on. The representative falls back to the existing process by looking up the answer manually or escalating it to the back-office helpdesk.

## Data Description

**Policy Documents**

| Document | Pages | Scope |
| :- | :-: | :- |
| Booking terms & cancellation | 5 | Rates, cancellation deadlines, no-shows, changes, taxes, deposits, property matrix |
| Room & property information | 5 | Rooms, arrival/departure, amenities, parking, pets, service animals, accessibility |
| Refunds & additional charges | 4 | Refund eligibility, charges, incidental holds, dispute workflow |
| Guest conduct, accessibility & resolution | 5 | Conduct, ADA/service animals, safety, complaints, escalation |

**Evaluation and Test Data Files**

| File | Queries | Scope |
| :- | :-: | :- |
| `evaluation.csv` | 10 | Dev benchmark: booking, property, conduct, and refund questions |
| `test.xlsx` | 10 | Held-out test: booking, property, conduct, and refund questions |

# **Please read the instructions carefully before starting the project.**

This is a commented Python notebook file in which all the instructions and tasks to be performed are mentioned.

* A significant portion of the code required to conduct the analysis and build and evaluate the predictive models are pre-written in the notebook, and only need to be executed.
* In certain sections, blanks '\_\_\_\_\_' are provided in the notebook that
needs to be filled appropriately to get the correct result. With every '\_\_\_\_\_' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Guidelines are also provided in the notebook on how to fill in these blanks. Read and follow these guidelines carefully before filling in any of the blanks.
* In certain sections, mutliple lines of code are provided and commented out. The appropriate code snippet needs to be uncommented before executing the respective code cells.
* Identify the task to be performed correctly, and only then proceed to fill in the blank or uncomment code snippets.
* Sequentially run the code cells from the beginning to avoid any unnecessary errors.
* Add the results/observations derived from the analysis in the presentation and submit the same. Any mathematical or computational details that are a graded part of the project can be included in the Appendix section of the presentation.

# Installing and Importing the Necessary Libraries

In [1]:
# Purpose: Install the pinned packages required for the Aurora Stays RAG workflow.
%pip install docling==2.126.0 pandas==2.2.3 transformers==5.16.1 langchain-huggingface==1.2.2 langchain-text-splitters==1.1.2 langchain-community==0.4.2 faiss-cpu==1.15.0 langchain-openai==1.6.1 deepeval==4.2.2 PyGithub==2.6.1 -q
# Code cell 001
# Install the required packages into the active Python/Jupyter environment (no virtual environment is used).


Note: you may need to restart the kernel to use updated packages.


**Output summary:** Note: you may need to restart the kernel to use updated packages.

**Note**:
- After running the above cell, kindly restart the runtime (for local Jupyter) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# Code cell 002
# Purpose: Import the notebook dependencies and configure project-wide runtime settings.
# Import function for displaying outputs in Jupyter/Google Colab
from IPython.display import display
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
HF_CACHE = Path(os.environ.get("TEMP", str(PROJECT_ROOT))) / "aurora-hf-cache"
os.environ.setdefault("HF_HOME", str(HF_CACHE))
os.environ.setdefault("HF_HUB_CACHE", str(HF_CACHE / "hub"))
import truststore
truststore.inject_into_ssl()

# Import library for working with tabular data
import pandas as pd

# To supress warnings
import warnings

# To make API requests
import requests

# Import Docling's document converter for processing PDF documents
from docling.document_converter import DocumentConverter

# Import tokenizer for determining token counts and preparing text for chunking
from transformers import AutoTokenizer

# Import LangChain's recursive character text splitter for token-aware chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Import library for working with JSON data and OS related functionalities
import json,os

# Import LangChain's Document class for representing text and metadata
from langchain_core.documents import Document as LCDocument

# Import Hugging Face embedding model integration for creating vector embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Import OpenAI-compatible chat model for generating RAG responses
from langchain_openai import ChatOpenAI

# Import FAISS vector database for storing and retrieving document embeddings
from langchain_community.vectorstores import FAISS

# Import DeepEval evaluation framework
from deepeval import evaluate

# Import DeepEval test case class for defining evaluation inputs and outputs
from deepeval.test_case import LLMTestCase

# Import DeepEval metrics for evaluating different aspects of the RAG system

from deepeval.test_case import SingleTurnParams
from deepeval.metrics import GEval

# Import OpenAI model wrapper used as the evaluation/judge model in DeepEval
from deepeval.models import OpenAIModel

# Import Python's operating system interface for managing environment variables
import os

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
warnings.filterwarnings("ignore", category=DeprecationWarning)

C:\Users\rvutu\AppData\Local\Temp\ipykernel_19172\3631829674.py:47: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


**Output summary:** C:\Users\rvutu\AppData\Local\Temp\ipykernel_19172\3631829674.py:47: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/

All the key libraries are imported here. The main components are:

- **Docling** for parsing PDF documents into structured markdown.
- **LangChain** modules for text splitting, embeddings, vector storage, and LLM interaction.
- **DeepEval** for evaluating the quality of the RAG pipeline using LLM-as-a-Judge metrics.
- **Pandas display options** are set to show full row/column content without truncation, which helps when inspecting long text chunks.
- **DeprecationWarnings** are suppressed to keep the notebook output clean.

# Data Understanding

## Loading the documents

- We define the file paths to the four hotel policy PDF documents that form the knowledge base for this RAG system.
- Each document covers a different area of hotel operations.

In [3]:
# Code cell 003
# Purpose: Resolve the project root and prepare the local data directory.
# Resolve the project root whether the notebook is opened from the repository root or notebooks/
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
root_path = str(PROJECT_ROOT / "data")
os.makedirs(root_path, exist_ok=True)


**Output summary:** No displayed output; cell completed successfully.

**Note:** After executing the previous cell, a folder named **Data** will be created. Ensure that all the PDF files are uploaded to this folder before executing the code cell below.

In [4]:
# Code cell 004
# Purpose: Define the four Aurora Stays policy PDFs used as the knowledge base.
# List all policy document PDF file paths
pdf_paths = [
    os.path.join(root_path, "aurorastay_room_and_property_information.pdf"),
    os.path.join(root_path, "aurorastay_refunds_and_additional_charges_guide.pdf"),
    os.path.join(root_path, "aurorastay_guest_conduct_accessibility_and_resolution_guidelines.pdf"),
    os.path.join(root_path, "aurorastay_booking_terms_and_cancellation_policy.pdf"),
]


**Output summary:** No displayed output; cell completed successfully.

Four policy documents are loaded:

1. **Room and property information** covering room types, amenities, and property-specific details.
2. **Refunds and additional charges** explaining fee structures and refund eligibility.
3. **Guest conduct, accessibility, and resolution guidelines** for behavioral policies and dispute handling.
4. **Booking terms and cancellation policy** with reservation rules and cancellation windows.

Together, these documents should cover the majority of questions hotel staff receive from guests.

- The `DocumentConverter` from Docling handles PDF parsing.
- It extracts text, tables, and document structure from PDFs and converts them into a format that preserves headings, lists, and table layouts.
  - This is important because hotel policy documents often contain structured tables (e.g., room rate grids, fee schedules) that would be lost with a simple text extractor.

In [5]:
# Code cell 005
# Purpose: Initialize Docling with OCR disabled for selectable-text policy PDFs.
# Initialize the Docling document converter.
# The supplied policy PDFs contain selectable text, so OCR is disabled to
# avoid the 'No OCR engine found' error when OCR extras are unavailable.
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption

pdf_pipeline_options = PdfPipelineOptions()
pdf_pipeline_options.do_ocr = False
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_pipeline_options)
    }
)

**Output summary:** No displayed output; cell completed successfully.

- Each PDF is converted one at a time and stored in a dictionary keyed by its file path.
- The converter processes the document layout, extracts text and tables, and returns a Docling document object.
- The print statement after each file confirms that the conversion completed successfully.
  - This step can take a minute or so depending on the size and complexity of the PDFs.

**Note:** The code snippet below may take 10 to 15 minutes to complete its execution.

In [6]:
# Code cell 006
# Purpose: Convert each policy PDF into a Docling document object.
# Convert each policy PDF into a Docling document object
docs = {}
for path in pdf_paths:
    result = converter.convert(path)
    docs[path] = result.document
    print(path," Completed ",end='\n')

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_room_and_property_information.pdf  Completed 


C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_refunds_and_additional_charges_guide.pdf  Completed 


C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_guest_conduct_accessibility_and_resolution_guidelines.pdf  Completed 


C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_booking_terms_and_cancellation_policy.pdf  Completed 


**Output summary:** Loading weights: 0%| | 0/770 [00:00<?, ?it/s] | C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_room_and_property_information.pdf Completed | C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_refunds_and_additional_charges_guide.pdf Completed | C:\Users\rvutu\OneDrive\Documents\RV\Le

## Viewing the documents

### Viewing the markdown

In [7]:
# Code cell 007
# Purpose: Select a policy document for the Markdown preview.
path = str(Path(root_path) / "aurorastay_room_and_property_information.pdf")
print(f"Preview document: {path}")


Preview document: C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_room_and_property_information.pdf


**Output summary:** Preview document: C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_room_and_property_information.pdf

In [8]:
# Code cell 008
# Display the converted document as markdown to inspect its preserved structure
print(docs[path].export_to_markdown())

## Room and Property Information Guide

Room types, arrival and departure, amenities, parking, breakfast, pets, and accessibility

| Policy code    | AS-02                                                                                                                                 |
|----------------|---------------------------------------------------------------------------------------------------------------------------------------|
| Effective date | January 1, 2026                                                                                                                       |
| Review date    | January 1, 2027, or sooner if law, systems, or operating conditions change                                                            |
| Owner          | Guest Experience and Property Operations                                                                                              |
| Applies to     | AuroraStay managed and franchised properties in the United States, unless a

**Output summary:** ## Room and Property Information Guide Room types, arrival and departure, amenities, parking, breakfast, pets, and accessibility | Policy code | AS-02 | |----------------|-------------------------------------------------

- The markdown export shows how Docling preserved the original document structure.
- Headings, subheadings, bullet points, and tables are all retained in markdown format.
- This is the text that will later be split into chunks for the vector database, so having clean, well-structured markdown directly improves retrieval quality.

### Viewing the tables

Let us also take a look at the tables extracted from the documents. Tables are particularly important in hotel policy documents because they often contain structured data like fee schedules, room configurations, and property-specific details.

In [9]:
# Code cell 009
# Purpose: Extract and display policy tables to verify document structure.
# Extract and display all tables from the converted documents
for path, doc in docs.items():
    for item, _ in doc.iterate_items():
        if getattr(item, "label", "") == "table":
            print(f"\n--- Table from: {path} ---")
            print(item.export_to_markdown(doc=doc))


--- Table from: C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_room_and_property_information.pdf ---
| Policy code    | AS-02                                                                                                                                 |
|----------------|---------------------------------------------------------------------------------------------------------------------------------------|
| Effective date | January 1, 2026                                                                                                                       |
| Review date    | January 1, 2027, or sooner if law, systems, or operating conditions change                                                            |
| Owner          | Guest Experience and Property Operations                                                                                              |
| Applies to 

| Topic                | Guideline                                                                                                                                                    | Response if not followed                                                                               |
|----------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|
| Quiet hours          | Quiet hours are 10:00 PM to 7:00 AM local property time. Guests must keep noise at a level that does not unreasonably disturb others at all times.           | Contact the room when safe, document the complaint, and escalate repeat or serious disturbances.       |
| Smoking and vaping   | Smoking and vaping are prohibited in guest rooms and indoor public areas unless a property specifically desi

| Rate plan             | Payment                                                             | Cancel or change                                                                                                              | Staff action                                                                           |
|-----------------------|---------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------|
| Flexible              | Card guarantee; payment at hotel unless confirmation says otherwise | Cancel by 6:00 PM local property time one day before arrival. Later cancellation or no-show is one night plus applicable tax. | Quote the deadline in local property time and note the confirmation number.            |
| Advance Purchase      | Full stay charged at booking       

**Output summary:** --- Table from: C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_room_and_property_information.pdf --- | Policy  | | Topic | Guideline | Response if not followed | |----------------------|--------------------------------------------------------------------------------------------------------------------------------------------------- | | Rate plan | Payment | Cancel or change | Staff actio

- The loop iterates through every item in each document and filters for items labeled as "table."
- Each table is exported as markdown, preserving its row/column structure.
- This confirms that Docling successfully extracted tabular data from the PDFs, which is essential since many guest queries involve looking up specific values from these tables.

# Retrieval Augmented Generation System

## Chunking

### Choosing the embedding model

Here, the embedding model chosen is **"HarishMaths/Hotel-Policy-Embedding"**, which is fine-tuned on hotel policy documents. Hence, it is expected to provide better retrieval quality compared to other base models.

**Note:** We highly recommend **not changing the embedding model**, as the model has been specifically fine-tuned for hotel policy documents and is expected to provide better retrieval performance for this use case.

For more details about the embedding model, please refer to the [Hugging Face model page](https://huggingface.co/HarishMaths/Hotel-Policy-Embedding).

In [10]:
# Code cell 010
# Purpose: Load the domain-specific tokenizer and inspect its token limit.
# define the model name
EMBED_MODEL = "HarishMaths/Hotel-Policy-Embedding"

# Load the domain-specific embedding model tokenizer and check its maximum token limit
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)

print(f"Model max tokens: {tokenizer.model_max_length}")

Model max tokens: 128


**Output summary:** Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads. | Model max tokens: 128

- The model's maximum token length tells us the upper bound on how much text can be encoded into a single embedding vector.
-  This limit directly affects how we configure the chunking strategy in the next step.
- Chunks that exceed this limit would be silently truncated, potentially losing important information.

Before chunking, it helps to see how the tokenizer actually breaks text into tokens. This gives us a sense of the token-to-word ratio for hotel policy language.

**Guidelines**

- Enter a **hotel policy related question or sentence** in `sample`.
- Try questions about topics such as **pet policies, refunds, cancellations, additional charges, room information, or accessibility**.
- After entering the sentence, run the cell to see how the tokenizer breaks it into tokens.

In [11]:
# Code cell 011
# Enter your own sentence in the blank below to test how it is tokenized
sample = "What is the cancellation policy for flexible rates?"

tokens = tokenizer.tokenize(sample)
print(f"Sample: '{sample}'")
print(f"Tokens ({len(tokens)}): {tokens}")

Sample: 'What is the cancellation policy for flexible rates?'
Tokens (9): ['what', 'is', 'the', 'cancellation', 'policy', 'for', 'flexible', 'rates', '?']


**Output summary:** Sample: 'What is the cancellation policy for flexible rates?' Tokens (9): ['what', 'is', 'the', 'cancellation', 'policy', 'for', 'flexible', 'rates', '?']

- The sample sentence produces a specific number of tokens, and we can see how individual words get split into subwords.
- Knowing this ratio helps us reason about chunk sizes.
  - For example, a 512-token chunk will typically hold roughly 350 to 400 words of hotel policy text, which is enough to capture a complete policy section in most cases.

### Define the chunker

The `RecursiveCharacterTextSplitter` is initialized with the same tokenizer used by the embedding model. This ensures that chunk sizes are measured in the same token units the model expects.

**Guidelines**

- **`chunk_size`**: Choose a value that captures enough context from each section without including unrelated information. For policy documents, **400 to 600 tokens** is a good starting point.
- **`chunk_overlap`**: Keep the overlap smaller than the chunk size so that important context is not lost between chunks. Around **10 to 15% of the chunk size** is a good starting point. For example, for a chunk size of 500, you can use an overlap of **50 to 75 tokens**.

In [12]:
# Code cell 012
# Purpose: Create the token-aware recursive text splitter.
import os

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=int(os.getenv("CHUNK_SIZE", "512")),
    chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "64")),
    separators=["\n\n", "\n", "##", ". ", " ", ""]
)


**Output summary:** No displayed output; cell completed successfully.

### Chunking the documents

In [13]:
# Code cell 013
# Purpose: Split policy documents into enriched, metadata-aware retrieval chunks.
import re
from copy import deepcopy

raw_docs = [
    LCDocument(page_content=doc.export_to_markdown(), metadata={"source": path})
    for path, doc in docs.items()
]

def annotate_chunk(chunk):
    text = chunk.page_content
    codes = sorted(set(re.findall(r"ASH-[A-Z]{3}", text.upper())))
    headings = re.findall(r"^#{1,4}\s+(.+)$", text, flags=re.MULTILINE)
    section = headings[-1].strip() if headings else ""
    lower = text.lower()
    topic_map = {
        "room_and_property": ("breakfast", "room", "parking", "pet", "accessible", "property", "occupancy"),
        "booking_terms": ("cancellation", "advance purchase", "reservation", "no-show", "check-in"),
        "refunds": ("refund", "charge", "incidentals", "hold", "smoking", "parking"),
        "guest_conduct": ("service animal", "accessibility", "quiet hours", "conduct", "safety"),
    }
    topics = [topic for topic, terms in topic_map.items() if any(term in lower for term in terms)]
    enriched = deepcopy(chunk)
    enriched.metadata.update({"property_codes": codes, "section": section, "topics": topics})
    return enriched

base_chunks = splitter.split_documents(raw_docs)
atomic_rows = []
for raw in raw_docs:
    if "room_and_property" not in raw.metadata.get("source", ""):
        continue
    heading = "Property-specific matrix"
    for line in raw.page_content.splitlines():
        if line.startswith("#"):
            heading = line.lstrip("# ").strip()
        if line.strip().startswith("|") and re.search(r"ASH-[A-Z]{3}", line.upper()):
            property_code = re.search(r"ASH-[A-Z]{3}", line.upper()).group(0)
            atomic_rows.append(LCDocument(
                page_content=f"## {heading}\nProperty: {property_code}\n{line}",
                metadata={"source": raw.metadata["source"], "property_codes": [property_code], "topics": ["room_and_property"]},
            ))
if os.getenv("ATOMIC_PROPERTY_ROWS", "true").lower() == "true":
    base_chunks.extend(atomic_rows)
lc_docs = [annotate_chunk(chunk) for chunk in base_chunks]
print(f"Total chunks: {len(lc_docs)} (atomic rows enabled={os.getenv('ATOMIC_PROPERTY_ROWS', 'true')})")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (232 > 128). Running this sequence through the model will result in indexing errors


Total chunks: 37 (atomic rows enabled=false)


**Output summary:** [transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (232 > 128). Running this sequence through the model will result in indexing errors | Total chunks: 37 (atomic rows enabled=false)

- Each document's markdown export is first wrapped into a single LangChain `Document` object with its source path as metadata. Then the splitter breaks these into smaller chunks.
- The total chunk count tells us how many vectors we will store in the database. Each chunk retains the `source` metadata so we can trace any retrieved chunk back to its original document.

Let us preview the first 400 characters of each chunk to verify the split looks reasonable.

In [14]:
# Code cell 014
# Preview the text content of each generated chunk
for i, chunk in enumerate(lc_docs):
    print(f"\n--- Chunk {i} | {chunk.metadata['source']} ---")
    print(chunk.page_content[:400])
    print("..." if len(chunk.page_content) > 1000 else "")


--- Chunk 0 | C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_room_and_property_information.pdf ---
## Room and Property Information Guide

Room types, arrival and departure, amenities, parking, breakfast, pets, and accessibility

| Policy code    | AS-02                                                                                                                                 |
|----------------|------------------------------------------------------------------------------------------------
...

--- Chunk 1 | C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_room_and_property_information.pdf ---
Operating note: This is a business policy for guest service teams. It is not a substitute for a property license, emergency procedure, signed group agreement, or advice from q

**Output summary:** --- Chunk 0 | C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\data\aurorastay_room_and_property_information.pdf --- ## Room and

- Scanning through the chunk previews, we can check whether the splits are happening at logical boundaries.
- Good chunks should start and end at natural breakpoints like section headings or paragraph boundaries, rather than cutting sentences in half.
- The overlap between consecutive chunks should also be visible.

Let us also verify the metadata attached to each chunk.

In [15]:
# Code cell 015
# Preview the metadata of each generated chunk
for i, chunk in enumerate(lc_docs):
    print(f"\n--- Chunk {i} ---")
    print(chunk.metadata)


--- Chunk 0 ---


{'source': 'C:\\Users\\rvutu\\OneDrive\\Documents\\RV\\Learning and development\\University of Texas - AI ML Training\\Assignments\\Assignment 4\\Aurora Stays QnA\\data\\aurorastay_room_and_property_information.pdf', 'property_codes': [], 'section': 'Document map', 'topics': ['room_and_property', 'booking_terms', 'refunds', 'guest_conduct']}

--- Chunk 1 ---
{'source': 'C:\\Users\\rvutu\\OneDrive\\Documents\\RV\\Learning and development\\University of Texas - AI ML Training\\Assignments\\Assignment 4\\Aurora Stays QnA\\data\\aurorastay_room_and_property_information.pdf', 'property_codes': [], 'section': '1. Room types and occupancy', 'topics': ['room_and_property', 'booking_terms']}

--- Chunk 2 ---
{'source': 'C:\\Users\\rvutu\\OneDrive\\Documents\\RV\\Learning and development\\University of Texas - AI ML Training\\Assignments\\Assignment 4\\Aurora Stays QnA\\data\\aurorastay_room_and_property_information.pdf', 'property_codes': [], 'section': '2. Arrival, departure, and access', 'to

**Output summary:** --- Chunk 0 --- | {'source': 'C:\\Users\\rvutu\\OneDrive\\Documents\\RV\\Learning and development\\University of Texas - AI ML Training\\Assignments\\Assignment 4\\Aurora Stays QnA\\data\\aurorastay_room_and_property_information.pdf', 'pr

- Each chunk carries a `source` field pointing to the PDF it came from.
- This metadata is preserved through the vector store and will be returned alongside retrieved chunks during inference, allowing us to cite the source document in the final answer.

## Vector Database

### Loading the embedding model

In [16]:
# Code cell 016
# Purpose: Load the hotel-policy embedding model on CPU.
# Load the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,  # Pre-trained embedding model
    model_kwargs={"device": "cpu"},                  # Run the model on CPU
    encode_kwargs={"normalize_embeddings": True},   # Normalize embeddings for similarity search
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**Output summary:** Loading weights: 0%| | 0/103 [00:00<?, ?it/s]

- The embedding model is loaded onto the CPU and configured to normalize its output vectors.
- Normalization ensures that cosine similarity and inner product give the same ranking, which makes the similarity search in FAISS more consistent.
- Since this is a domain-specific model fine-tuned on hotel policy data, it should produce better embeddings for this use case compared to a general-purpose model.

### Creating the Vector Database

**Note:** The code snippet below may take approximately **4-5 minutes** to complete.

In [17]:
# Code cell 017
# Purpose: Build the FAISS vector store from the policy chunks.
# Create a FAISS vector store from the LangChain documents
vectorstore = FAISS.from_documents(
    documents=lc_docs,
    embedding=embeddings,
)
print(f"Vectors stored: {vectorstore.index.ntotal}")

Vectors stored: 37


**Output summary:** Vectors stored: 37

- FAISS builds an in-memory index from the chunk embeddings.
- The `ntotal` count confirms that every chunk was successfully embedded and stored.
- At query time, FAISS will compare the query embedding against all stored vectors and return the closest matches.
- Because the index lives in memory, retrieval is very fast even for hundreds of chunks.

### Retrieval System

The retrieval function wraps FAISS similarity search into a clean interface. Given a query string, it returns the top-k most similar chunks along with their metadata.

In [18]:
# Code cell 018
# Purpose: Define hierarchical retrieval and format retrieved chunks for generation.
from backend.rag_utils import retrieve_documents as hierarchical_retrieve_documents

def retrieve(query: str, k: int = 5, vectorstore=None) -> list[str]:
    results = hierarchical_retrieve_documents(query=query, k=k, vectorstore=vectorstore)
    return [
        f"Content:\n{r.page_content}\n\nMetadata:\n{r.metadata}"
        for r in results
    ]


**Output summary:** No displayed output; cell completed successfully.

- The `retrieve` function performs a similarity search on the vector store and returns formatted strings containing both the chunk content and its metadata.
- Formatting them this way makes it easy to pass the results directly into the LLM prompt later.
- The `k` parameter controls how many chunks are retrieved, defaulting to 2.

Let us test the retrieval system with a sample query about pet charges at a specific property.

**Guidelines**

- Enter a **hotel policy related question or query** in `query` to test the retrieval. You can ask about topics such as **pet policies, refunds, cancellations, additional charges, room information, or accessibility**.
- Set `k` to the number of relevant chunks you want to retrieve. Start with **2 or 3** and increase it if the relevant information is not retrieved.

In [19]:
# Code cell 019
# Test the retrieval results

query = "What are the pet fees at Aurora Stays properties?"  # Enter a question or query to test the retrieval
k = 5        # Baseline number of chunks to retrieve

for i, chunk in enumerate(retrieve(query=query, k=k, vectorstore=vectorstore)):
    print(f"\n--- Result {i} ---")
    print(chunk)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


--- Result 0 ---
Content:
## 3. Accessibility and service animals

AuroraStay follows the Americans with Disabilities Act Title III requirements for places of lodging and provides reasonable modifications and accessible features as required by applicable law. Staff should focus on the requested feature or task, not on the guest's diagnosis or private medical information.

| Question or issue                                     | Required handling                                                                                                                                             |
|-------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------|
| A guest requests an accessible feature                | Ask what feature is needed, check the room and property profile, record the request, and confirm what can be provided. Escalate if

**Output summary:** Loading weights: 0%| | 0/105 [00:00<?, ?it/s] | --- Result 0 --- Content: ## 3. Accessibility and service animals AuroraStay follows the Americans with Disabilities Act Title III requirements for places of lodging and provides reasonable modifications and accessible f

- The retrieved chunks contain information about pet policies and charges.  
- Looking at the results, we can verify whether the retrieval system is pulling in the right sections from the knowledge base.
- If the chunks are relevant to the query, it means the embedding model and vector store are working together correctly.

## Generation System

### Loading the LLM

We load the OpenAI API credentials from the local `config/config.json` file for use with the LLM and embedding model.

**Note:** Copy `config/config.example.json` to `config/config.json` and fill in your own credentials before running this cell.

In [20]:
# Code cell 020
# Purpose: Load local API configuration and set retrieval/runtime environment variables.
# Load local API configuration, with the supplied assignment config as a fallback.
CONFIG_PATH = PROJECT_ROOT / "config" / "config.json"
SUPPLIED_CONFIG_PATH = PROJECT_ROOT / "Supplied files" / "confignewproject4.txt"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)
if not config.get("OPENAI_API_KEY") and SUPPLIED_CONFIG_PATH.exists():
    with open(SUPPLIED_CONFIG_PATH, "r", encoding="utf-8-sig") as f:
        supplied_config = json.load(f)
    config.update({k: v for k, v in supplied_config.items() if v})

os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]
os.environ["OPENAI_BASE_URL"] = config.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
os.environ["LANGCHAIN_TRACING_V2"] = config.get("LANGCHAIN_TRACING_V2", "true")
os.environ["LANGCHAIN_API_KEY"] = config.get("LANGCHAIN_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = "Aurora Stays QnA K5 MMR Rerank Trial"
os.environ["LANGCHAIN_ENDPOINT"] = "https://apac.api.smith.langchain.com"
os.environ["ADAPTIVE_K"] = "false"
os.environ["HYBRID_RETRIEVAL"] = "false"

print(f"Configuration loaded from {CONFIG_PATH} with supplied fallback")


Configuration loaded from C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\config\config.json with supplied fallback


**Output summary:** Configuration loaded from C:\Users\rvutu\OneDrive\Documents\RV\Learning and development\University of Texas - AI ML Training\Assignments\Assignment 4\Aurora Stays QnA\config\config.json with supplied fallback

- The API key and base URL are loaded from the local config file and set as environment variables.

In [21]:
# Code cell 021
# Purpose: Configure the OpenAI-compatible answer-generation model.
# Define the model configuration
MODEL = "gpt-4o-mini"
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_TOKENS = 512

llm = ChatOpenAI(
    model=MODEL,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
    openai_api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"]
)


**Output summary:** No displayed output; cell completed successfully.

**Guidelines**

- Enter a **short and simple prompt** in `TEST_QUERY_LLM` to check whether the LLM is connected and responding correctly.
- You can ask the model to return a specific short response so that it is easy to verify the output.
- This is only a connectivity test, so there is no need to use a hotel policy question here.

In [22]:
# Code cell 022
# Define a simple test query to check whether the LLM is working correctly
TEST_QUERY_LLM = "Reply with exactly: LLM connection successful."
response = llm.invoke(TEST_QUERY_LLM)
print(response.content)


LLM connection successful.


**Output summary:** LLM connection successful.

- A quick connectivity check to make sure the LLM is reachable and responding.
- If the output says "LLM is ready," it confirms that the API key, base URL, and model name are all configured correctly.

### Defining the generation system

The generation function builds a structured prompt that includes the user's query and the retrieved context, along with a set of strict rules. These rules are critical for the RAG system's reliability:

- Rules 1, 2, and 9 enforce **faithfulness** by restricting the model to only use information from the retrieved context.
- Rules 3 and 4 ensure **completeness** by requiring the model to include all relevant details.
- Rules 5, 6, and 7 enforce **relevancy** by keeping the answer focused on the actual question.
- Rule 10 provides a **fallback** so the model gracefully handles cases where the context is insufficient.

This prompt design directly supports the evaluation metrics we defined in the success criteria.

**Guidelines**

- The prompt provided below is a **baseline prompt** for generating answers from the retrieved hotel policy information.
- You can **modify the prompt if needed** to improve the quality, relevance, or clarity of the responses.
- Make sure the modified prompt still instructs the LLM to use only the retrieved policy information and avoid making up information.
- You can experiment with the instructions and observe how changes affect the generated answers.

In [23]:
# Code cell 023
# Purpose: Define the grounded prompt used to generate policy answers.
# Define the generation function that prompts the LLM with retrieved context
def generate(query, retrieved_chunks, model) -> str:

    prompt = f"""
    You are an AI assistant answering questions about Aurora Stays hotel policies.

    User Query:
    {query}

    Retrieved Policy Information:
    {retrieved_chunks}

    Rules:
    1. Use only information explicitly supported by the retrieved policy information.
    2. Do not use outside knowledge, assumptions, or information not present in the retrieved policy information.
    3. Include all relevant information needed to answer the query completely.
    4. Do not omit important conditions, exclusions, limitations, exceptions, or coverage details.
    5. Focus only on information directly relevant to the user's query.
    6. Do not include unrelated information from the retrieved policy information.
    7. Answer the user's specific question directly and ensure the response addresses what was asked.
    8. Keep the answer concise, clear, and easy to understand.
    9. Do not invent benefits, exclusions, limits, waiting periods, conditions, or coverage details.
    10. If the information is insufficient, say:
        "The available policy documents do not contain enough information to answer this question."
        If the retrieved information conflicts, clearly mention the conflict.

    Answer:
    """

    return model.invoke(prompt)

**Output summary:** No displayed output; cell completed successfully.

### Defining the RAG Function

- The `rag` function ties the retrieval and generation steps together into a single callable pipeline.
- It creates a fresh LLM instance with the specified parameters, retrieves the top-k chunks, and passes them to the generation function.
- It returns both the generated answer and the retrieved chunks, which is useful for evaluation and for displaying source information in the UI.

In [24]:
# Code cell 024
# Purpose: Assemble retrieval and generation into the reusable RAG function.
def rag(
    query: str,
    k: int = 5,
    model_name: str = "gpt-4o-mini",
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_tokens: int = 512,
    vectorstore=None
):
    model = ChatOpenAI(
        model=model_name,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens
    )
    retrieved_chunks = retrieve(query=query, k=k, vectorstore=vectorstore)
    answer = generate(
        query=query,
        retrieved_chunks=retrieved_chunks,
        model=model
    )
    return answer.content, retrieved_chunks


**Output summary:** No displayed output; cell completed successfully.

Let us run the full RAG pipeline end to end with the same pet charge query to see the complete flow in action.

**Guidelines**

- Enter a **hotel policy related question** in `query` to test the complete RAG pipeline.
- Try questions about topics such as **pet policies, refunds, cancellations, additional charges, room information, or accessibility**.
- You can also try questions that mention a **specific property, charge, condition, or policy**.
- After running the cell, review the response and check whether it correctly answers the question using the retrieved policy information.

In [25]:
# Code cell 025
# Test the RAG pipeline with a sample query
query = "What is the cancellation policy for a flexible reservation?"
response, rel_chunks = rag(query, vectorstore=vectorstore)


**Output summary:** No displayed output; cell completed successfully.

In [26]:
# Code cell 026
# Print the generated response
print("Response: ",response)

Response:  For a flexible reservation at Aurora Stays, you can cancel without a room charge by 6:00 PM local property time one day before your arrival. If you cancel later or do not show up, you will be charged for one night plus applicable tax.


**Output summary:** Response: For a flexible reservation at Aurora Stays, you can cancel without a room charge by 6:00 PM local property time one day before your arrival. If you cancel later or do not show up, you will be charged for one ni

- The generated response should directly answer the pet charge question using only information from the retrieved chunks.
- If the answer references specific dollar amounts, property names, or policy conditions, those details should all be traceable back to the retrieved context.
- This is exactly the kind of grounded, source-backed answer the system is designed to produce.

# Evaluation Pipeline

## Loading the Evaluation Data

- The evaluation dataset is a CSV file that contains pre-written questions, expected answers, and the relevant context that a correct answer should be based on.
- This lets us systematically measure how well the RAG pipeline performs.

**Note:** Before executing the code cell below, ensure that the `evaluation.csv` and `test.csv` files are uploaded to the `Data` folder.

In [27]:
# Code cell 027
# Load evaluation data
df = pd.read_csv("data/evaluation.csv")

**Output summary:** No displayed output; cell completed successfully.

- The evaluation CSV is loaded into a DataFrame.
- Each row represents one test case with columns for the question, the expected answer, and the relevant context passages.
- These will be compared against the RAG pipeline's actual outputs during the evaluation step.

## Initializing the Evaluation Model

For evaluation, we use a separate, stronger model (`gpt-4o`) as the judge. Using a different model for evaluation avoids the bias that would come from having the same model grade its own answers.

In [28]:
# Code cell 028
# Purpose: Initialize the evaluation model used by DeepEval.
# Initialize the evaluation model using a stronger LLM for scoring

EVAL_MODEL = 'gpt-4o'

eval_model = OpenAIModel(
    model=EVAL_MODEL,
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"]
)

**Output summary:** No displayed output; cell completed successfully.

**Guidelines**

- Enter a short and simple prompt to check whether the evaluation model is responding correctly.
- This is only a connectivity test, so you do not need to use a hotel policy question.
- Once the model responds successfully, you can proceed with the evaluation.

In [29]:
# Code cell 029
# Verify that the evaluation model is responding correctly
eval_model.generate("Reply with exactly: Evaluation model connection successful.")


('Evaluation model connection successful.', 9e-05)

**Output summary:** ('Evaluation model connection successful.', 9e-05)

- The evaluation model is responding correctly.

- We use a stronger model (`gpt-4o`) for evaluation scoring, separate from the generation model (`gpt-4o-mini`), to ensure that the evaluation itself is reliable and not biased by the same model that generated the answers.

## Defining Evaluation Metrics

Four metrics are defined, each corresponding to one of the risks identified in the problem statement.
- **Faithfulness** guards against misrepresentation
- **Contextual Recall** checks retrieval completeness
- **Contextual Precision** validates that retrieved chunks are relevant and verifiable
- **Answer Relevancy** ensures the response addresses the actual question.

In [30]:
# Code cell 030
# Purpose: Define faithfulness, recall, precision, and answer-relevancy metrics.
# Faithfulness
# Checks whether the generated answer is factually supported by
# the information retrieved from the knowledge base.
faithfulness_metric = GEval(
    name="Faithfulness",

    # Defines what the evaluator should judge.
    criteria=(
        "Determine whether the actual output is factually supported by "
        "the provided retrieval context."
    ),

    # Step-by-step instructions given to the evaluator.
    evaluation_steps=[
        # First, identify the individual factual claims made in the answer.
        "Break the actual output into individual factual claims.",

        # Check each claim against the retrieved context.
        "Check whether each claim is supported by the retrieval context.",

        # Lower the score if the answer contains unsupported or conflicting claims.
        "Penalize claims that are unsupported or contradicted by the retrieval context.",

        # Differences in wording should not affect the score if the meaning is supported.
        "Do not penalize differences in wording when the meaning is supported by the context.",

        # Give a high score when the important claims are supported.
        "Give a higher score when all important claims are supported by the context."
    ],

    # Specifies which parts of the test case are used for evaluation.
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT
    ],

    # Minimum score required for the test case to pass.
    threshold=0.85,

    # LLM used to perform the evaluation.
    model=eval_model
)


# Contextual Recall
# Checks whether the retrieved context contains the information
# needed to produce the expected answer.
recall_metric = GEval(
    name="Contextual Recall",

    # Defines what the evaluator should judge.
    criteria=(
        "Determine whether the retrieval context contains the information "
        "needed to support the expected output."
    ),

    # Step-by-step instructions given to the evaluator.
    evaluation_steps=[
        # Identify the important claims that the expected answer should contain.
        "Break the expected output into its important factual claims.",

        # Check whether those claims can be found or supported by the retrieved context.
        "Check whether each important claim is present or supported by the retrieval context.",

        # Lower the score when important information is missing from retrieval.
        "Penalize the retrieval context when important information required for the expected output is missing.",

        # Wording differences should not affect the evaluation.
        "Do not penalize differences in wording or presentation.",

        # Give a high score when retrieval contains most or all required information.
        "Give a higher score when the retrieval context contains most or all information needed to produce the expected output."
    ],

    # Recall requires the expected answer and retrieved context.
    evaluation_params=[
        SingleTurnParams.EXPECTED_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT
    ],

    # Minimum score required for the test case to pass.
    threshold=0.80,

    # LLM used to perform the evaluation.
    model=eval_model
)


# Contextual Precision
# Checks whether the retrieved context is relevant to the user's question
# and whether relevant information is prioritized over irrelevant information.
precision_metric = GEval(
    name="Contextual Precision",

    # Defines what the evaluator should judge.
    criteria=(
        "Determine whether the retrieved context is relevant to answering "
        "the input question, with relevant information prioritized over irrelevant information."
    ),

    # Step-by-step instructions given to the evaluator.
    evaluation_steps=[
        # Identify which parts of the retrieved context are useful for the question.
        "Identify the information in the retrieval context that is relevant to the input question.",

        # Check whether unnecessary or unrelated information was retrieved.
        "Check whether the retrieval context contains unnecessary or unrelated information.",

        # Lower the score when a large portion of the retrieved context is irrelevant.
        "Penalize retrievals that contain a large amount of irrelevant information.",

        # Give more credit when useful information appears prominently.
        "Give more credit when relevant information is prioritized.",

        # Give a high score when most retrieved information is useful.
        "Give a higher score when most of the retrieved context is useful for answering the question."
    ],

    # Precision evaluates the question against the retrieved context.
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT
    ],

    # Minimum score required for the test case to pass.
    threshold=0.75,

    # LLM used to perform the evaluation.
    model=eval_model
)


# Answer Relevancy
# Checks whether the generated answer directly addresses the user's question
# without including unnecessary or unrelated information.
answer_relevancy_metric = GEval(
    name="Answer Relevancy",

    # Defines what the evaluator should judge.
    criteria=(
        "Determine whether the actual output directly addresses the input "
        "question and provides an appropriate response without unnecessary information."
    ),

    # Step-by-step instructions given to the evaluator.
    evaluation_steps=[
        # Check whether the answer directly responds to the question.
        "Check whether the actual output directly answers the input question.",

        # Check whether the information provided is relevant to the question.
        "Check whether the response contains information relevant to the question.",

        # Lower the score for unrelated or unnecessary information.
        "Penalize unnecessary, unrelated, or off-topic information.",

        # A short answer should not be penalized if it answers the question sufficiently.
        "Do not penalize concise answers when they adequately address the question.",

        # Give a high score when the answer is focused and directly relevant.
        "Give a higher score when the response is focused and directly relevant to the question."
    ],

    # Answer relevancy evaluates the question against the generated answer.
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],

    # Minimum score required for the test case to pass.
    threshold=0.75,

    # LLM used to perform the evaluation.
    model=eval_model
)


# Store all evaluation metrics in a single list.
# This list can then be passed to DeepEval's evaluation function.
metrics = [
    faithfulness_metric,
    recall_metric,
    precision_metric,
    answer_relevancy_metric
]

**Output summary:** No displayed output; cell completed successfully.

Four evaluation metrics are defined, each targeting a specific aspect of RAG quality:

- **Faithfulness** (threshold >= 0.85): Checks whether the generated answer is factually supported by the retrieved context. Uses `ACTUAL_OUTPUT` and `RETRIEVAL_CONTEXT`.
- **Contextual Recall** (threshold >= 0.80): Checks whether the retrieved context contains the information needed to produce the expected answer. Uses `EXPECTED_OUTPUT` and `RETRIEVAL_CONTEXT`.
- **Contextual Precision** (threshold >= 0.75): Checks whether the retrieved chunks are relevant to the question and whether useful information is prioritized. Uses `INPUT` and `RETRIEVAL_CONTEXT`.
- **Answer Relevancy** (threshold >= 0.75): Checks whether the answer directly addresses the user's question without unnecessary information. Uses `INPUT` and `ACTUAL_OUTPUT`.

All four metrics use the GEval framework, which provides step-by-step evaluation instructions to the judge model. The thresholds match the success criteria defined in the problem statement.

## Building Evaluation Test Cases

The function below takes the evaluation DataFrame and runs the RAG pipeline on each question. It then packages the inputs, outputs, and ground-truth data into DeepEval test cases for scoring.

In [31]:
# Code cell 031
# Purpose: Create evaluation test cases from the held-out questions and RAG outputs.
def test_case_generator(df,vectorstore,k=5,temperature=0,top_p=1,max_tokens=512):

    # Store the generated DeepEval test cases
    test_cases = []

    # Process each evaluation question
    for _, row in df.iterrows():

        # Run the RAG pipeline
        answer, retrieved_chunks = rag(
            query=row["question"],
            k=k,
            model_name=MODEL,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
            vectorstore=vectorstore
        )

        # Split the ground-truth relevant context
        # Multiple contexts are separated by "<>"
        relevant_context = [
            c.strip()
            for c in row["relevant_context"].split("<>")
            if c.strip()
        ]

        # Create the DeepEval test case
        test_case = LLMTestCase(
            # User question
            input=row["question"],

            # Answer generated by the RAG pipeline
            actual_output=answer,

            # Ground-truth expected answer
            expected_output=row["expected_answer"],

            # Context actually retrieved by the RAG pipeline
            retrieval_context=retrieved_chunks,

            # Ground-truth relevant context
            context=relevant_context
        )

        # Add test case
        test_cases.append(test_case)

    return test_cases

**Output summary:** No displayed output; cell completed successfully.

For each row in the evaluation data, the function:

1. Runs the full RAG pipeline (retrieve + generate) on the question.
2. Splits the ground-truth relevant context on the `<>` delimiter, since some questions need multiple context passages.
3. Builds an `LLMTestCase` containing the input question, the generated answer, the expected answer, the retrieved context, and the ground-truth context.

This structure gives DeepEval everything it needs to score each metric.

## Running the Evaluation

The below step might take a few minutes because it runs the RAG pipeline on every question in the evaluation dataset. Each call involves embedding the query, searching the vector store, and generating an answer with the LLM.

**Guidelines**

- The code below creates test cases and evaluates the RAG pipeline using the selected metrics.
- Choose `k` based on how much retrieved context you want to provide for each hotel policy question. A value around **2 to 5** is a good starting point.
- Use a low `temperature`, such as **0 to 0.2**, for more consistent evaluation responses.
- Keep `top_p` around **0.9 to 1** for consistent responses.
- Choose `max_tokens` based on the expected answer length. **256 to 512** is a reasonable starting range for hotel policy questions.
- You can experiment with these values and compare how they affect the evaluation results.

In [ ]:
# Code cell 032
# Define the parameters for test case generation
K = 5
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_TOKENS = 512

test_cases = test_case_generator(
    df,
    vectorstore,
    k=K,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS
)

results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    async_config=__import__("deepeval.evaluate.configs", fromlist=["AsyncConfig"]).AsyncConfig(
        run_async=False
    )
)


**Note:** The evaluation results above are primarily obtained using an LLM-as-a-Judge approach. Since LLM-based evaluation can be non-deterministic, the metric scores and observations may vary slightly across different evaluation runs.

Therefore, the results from a single run should not be treated as definitive. Running the evaluation multiple times and looking at the overall trends can provide a more reliable estimate of the system's performance.

The function below converts raw evaluation results into a structured DataFrame with composite scores and routing decisions.

In [ ]:
# Code cell 033
# Purpose: Run the evaluation suite and collect metric results.
def create_evaluation_df(test_cases, results):

    rows = []

    # Loop through each test case and its corresponding evaluation result
    for test_case, result in zip(test_cases, results.test_results):

        # Create a dictionary of metric name -> score
        # Remove "[GEval]" from the metric names so they match our keys
        scores = {
            metric.name.replace(" [GEval]", "").strip().lower(): metric.score
            for metric in result.metrics_data
        }

        # Calculate the weighted composite score
        composite = (
            0.35 * scores["faithfulness"]
            + 0.25 * scores["contextual recall"]
            + 0.20 * scores["contextual precision"]
            + 0.20 * scores["answer relevancy"]
        )

        # Determine the routing type based on the composite score
        if composite > 0.75:
            routing_type = "AUTO ANSWER"
        elif composite >= 0.50:
            routing_type = "ASSISTED REVIEW"
        else:
            routing_type = "MANUAL LOOKUP"

        # Store the evaluation results for this question
        rows.append({
            "Question": test_case.input,
            "Faithfulness Score": scores["faithfulness"],
            "Contextual Recall Score": scores["contextual recall"],
            "Contextual Precision Score": scores["contextual precision"],
            "Answer Relevancy Score": scores["answer relevancy"],
            "Composite Score": composite,
            "Type": routing_type
        })

    return pd.DataFrame(rows)

The composite score calculation uses the weights from the success criteria: Faithfulness gets the highest weight (35%) because avoiding unsupported information is the top priority. Contextual Recall (25%) is next, followed by Contextual Precision (20%) and Answer Relevancy (20%).

The routing thresholds determine how each query is handled in production:
- Scores above 0.75 are reliable enough for automatic answers.
- Scores between 0.50 and 0.75 need human review.
- Scores below 0.50 require full manual lookup.

This tiered approach lets the system handle easy queries automatically while flagging uncertain ones for human attention.

In [ ]:
# Code cell 034
# Generate the evaluation results DataFrame with composite scores and routing decisions
create_evaluation_df(test_cases, results)

- The output table shows each evaluation question alongside its individual metric scores, the weighted composite score, and the routing decision.
- This gives a clear picture of where the system is performing well and where it needs improvement.
- Questions routed to "AUTO ANSWER" are the ones the system can handle confidently without human intervention.
- As mentioned in the objective, **at least 50% of the queries should be routed to "AUTO ANSWER"**

# Evaluating on the Test set

## Loading the Test Data

Now we load a separate test dataset to evaluate the system on questions it has not been tuned against. This gives a more honest estimate of real-world performance.

In [ ]:
# Code cell 035
# Purpose: Prepare the held-out test set and evaluation inputs.
# Load the supplied test data workbook
df = pd.read_excel("data/test.xlsx")


## Running the Evaluation

**Note**

The `K`, `TEMPERATURE`, `TOP_P`, and `MAX_TOKENS` values selected earlier for the evaluation setup are used here as well.

This keeps the retrieval and generation settings consistent, making the evaluation results comparable.

In [ ]:
# Code cell 036
# Purpose: Evaluate the held-out test set using the configured RAG and metrics.
# Use the same retrieval and generation settings for the held-out test set.
K = 5
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_TOKENS = 512

test_cases = test_case_generator(
    df,
    vectorstore,
    k=K,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS
)

results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    async_config=__import__("deepeval.evaluate.configs", fromlist=["AsyncConfig"]).AsyncConfig(
        run_async=False
    )
)


The same evaluation pipeline is applied to the test set. First, the RAG pipeline generates answers for all test questions, then DeepEval scores each one across the four metrics. This can take several minutes depending on the number of test cases and API response times.

In [ ]:
# Code cell 037
# Purpose: Display the evaluation results for review.
# Display the test set evaluation results with composite scores and routing decisions
create_evaluation_df(test_cases, results)

- The test set results show how the system performs on unseen questions.
- Comparing these scores to the evaluation set results helps identify whether the system generalizes well or if there is overfitting to the evaluation data. - Ideally, the distribution of AUTO ANSWER vs. MANUAL LOOKUP should be similar across both sets.

**Note**

The next two sections, **Deployment - Backend** and **Deployment - Frontend**, focus on setting up the backend and frontend applications.

We highly recommend **not modifying the code snippets** in these two sections. The code contains several implementation details, and even small changes can lead to errors during deployment.

A document titled **"Understanding Deployment"** is provided for those who would like to understand these components in more detail. A high-level understanding of the document is sufficient.

# Deployment - Backend

## Serialize the Vector Database

Before writing out the deployment files, we create a `backend` directory to hold all the backend components that will be pushed to GitHub and deployed in a Codespace.

In [ ]:
# Code cell 038
import os

# Create a folder to upload the backend files
os.makedirs("backend", exist_ok=True)

In [ ]:
# Code cell 039
# Save the index locally so it can be reloaded without rebuilding
vectorstore.save_local("backend/faiss_index")

- The FAISS index is serialized to disk so it can be loaded directly in the backend without rebuilding from scratch.
  - This saves significant time during deployment because embedding all chunks takes several minutes.
- The saved index includes both the vector data and the document metadata.

## Define the Flask application

- The `rag_utils.py` file contains the core RAG functions (retrieve, generate, and rag) that the Flask API will call.
- These are the same functions we defined and tested earlier in the notebook, now written to a standalone Python file for deployment.

In [ ]:
%%writefile backend/rag_utils.py
# Code cell 040

from langchain_openai import ChatOpenAI
import os

# Retrieve the top-k relevant chunks from the vector store
def retrieve(query: str, k: int = 2,vectorstore=None) -> list[str]:
    results = vectorstore.similarity_search(query, k=k)

    # Return each chunk with its content and metadata
    return [
        f"Content:\n{r.page_content}\n\nMetadata:\n{r.metadata}"
        for r in results
    ]

# Define the generation function that prompts the LLM with retrieved context
def generate(query, retrieved_chunks,model) -> str:

    prompt = f"""
    You are an AI assistant answering questions about Aurora Stays hotel policies.

    User Query:
    {query}

    Retrieved Policy Information:
    {retrieved_chunks}

    Rules:
    1. Use only information explicitly supported by the retrieved policy information.
    2. Do not use outside knowledge, assumptions, or information not present in the retrieved policy information.
    3. Include all relevant information needed to answer the query completely.
    4. Do not omit important conditions, exclusions, limitations, exceptions, or coverage details.
    5. Focus only on information directly relevant to the user's query.
    6. Do not include unrelated information from the retrieved policy information.
    7. Answer the user's specific question directly and ensure the response addresses what was asked.
    8. Keep the answer concise, clear, and easy to understand.
    9. Do not invent benefits, exclusions, limits, waiting periods, conditions, or coverage details.
    10. If the information is insufficient, say:
        "The available policy documents do not contain enough information to answer this question."
        If the retrieved information conflicts, clearly mention the conflict.

    Answer:
    """

    return model.invoke(prompt)

# Define the full RAG pipeline combining retrieval and generation
def rag(
    query: str,
    k: int = 2,
    model_name: str = "gpt-4o-mini",
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_tokens: int = 512,
    vectorstore=None
):
    # Create the generator model
    model = ChatOpenAI(
        model=model_name,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        openai_api_key=os.getenv('OPENAI_API_KEY'),
        base_url=os.getenv('OPENAI_BASE_URL')
    )

    # Retrieve relevant chunks
    retrieved_chunks = retrieve(query=query, k=k,vectorstore=vectorstore)

    # Generate answer using retrieved chunks
    answer = generate(
        query=query,
        retrieved_chunks=retrieved_chunks,
        model=model
    )

    return answer.content, retrieved_chunks

- This file is written to `backend/rag_utils.py` using the `%%writefile` magic command.
- The key difference from the notebook versions is that the `rag` function reads the API credentials from environment variables (`os.getenv`) instead of a config file, since the deployed environment will use GitHub Codespace secrets which will be discussed later.

- The Flask application defines the REST API endpoints that the frontend will call.
- It loads the embedding model and FAISS index at startup, then exposes three endpoints for different levels of response detail.

In [ ]:
%%writefile backend/app.py
# Code cell 041

# Import necessary libraries
from flask import Flask, request, jsonify
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Import RAG utility functions
from rag_utils import retrieve, rag


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

EMBED_MODEL = "HarishMaths/Hotel-Policy-Embedding"
VECTORSTORE_PATH = "faiss_index"


# ---------------------------------------------------------
# Initialize the Flask application
# ---------------------------------------------------------

rag_api = Flask("Aurora Stay API")


# ---------------------------------------------------------
# Load the embedding model
# ---------------------------------------------------------

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)


# ---------------------------------------------------------
# Load the saved FAISS vector store
# ---------------------------------------------------------

vectorstore = FAISS.load_local(
    VECTORSTORE_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Vectors stored: {vectorstore.index.ntotal}")


# ---------------------------------------------------------
# Default route
# ---------------------------------------------------------

@rag_api.get("/")
def home():
    """
    Handles GET requests to the root URL.
    """
    return "Welcome to the Aurora Stay RAG API!"


# ---------------------------------------------------------
# Relevant chunks endpoint
# ---------------------------------------------------------

@rag_api.post("/v1/relevant_chunks")
def relevant_chunks():
    """
    Handles POST requests to /v1/relevant_chunks.

    Example request:
    {
        "query": "What is the cancellation policy?",
        "k": 3
    }

    Returns the top-k relevant chunks retrieved from
    the vector store.
    """

    # Get JSON data from request body
    data = request.get_json()

    # Extract parameters
    query = data["query"]
    k = data.get("k", 5)

    # Retrieve relevant chunks
    chunks = retrieve(
        query=query,
        k=k,
        vectorstore=vectorstore
    )

    # Return the retrieved chunks
    return jsonify({
        "query": query,
        "k": k,
        "relevant_chunks": chunks
    })


# ---------------------------------------------------------
# Answer with relevant chunks endpoint
# ---------------------------------------------------------

@rag_api.post("/v1/answer_with_relevant_chunks")
def answer_with_relevant_chunks():
    """
    Handles POST requests to /v1/answer_with_relevant_chunks.

    Example request:
    {
        "query": "What is the cancellation policy?",
        "k": 3,
        "model_name": "gpt-4o-mini",
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 512
    }

    Returns both the generated answer and the relevant
    chunks used to generate the answer.
    """

    # Get JSON data from request body
    data = request.get_json()

    # Extract parameters
    query = data["query"]
    k = data.get("k", 5)
    model_name = data.get("model_name", "gpt-4o-mini")
    temperature = data.get("temperature", 0.0)
    top_p = data.get("top_p", 1.0)
    max_tokens = data.get("max_tokens", 512)

    # Run the complete RAG pipeline
    answer, chunks = rag(
        query=query,
        k=k,
        model_name=model_name,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        vectorstore=vectorstore
    )

    # Return answer, chunks, and parameters used
    return jsonify({
        "query": query,
        "k": k,
        "model_name": model_name,
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
        "relevant_chunks": chunks,
        "answer": answer
    })


# ---------------------------------------------------------
# Answer endpoint
# ---------------------------------------------------------

@rag_api.post("/v1/answer")
def answer():
    """
    Handles POST requests to /v1/answer.

    Example request:
    {
        "query": "What is the cancellation policy?",
        "k": 3,
        "model_name": "gpt-4o-mini",
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 512
    }

    Returns only the generated answer.
    """

    # Get JSON data from request body
    data = request.get_json()

    # Extract parameters
    query = data["query"]
    k = data.get("k", 5)
    model_name = data.get("model_name", "gpt-4o-mini")
    temperature = data.get("temperature", 0.0)
    top_p = data.get("top_p", 1.0)
    max_tokens = data.get("max_tokens", 512)

    # Run the complete RAG pipeline
    generated_answer, _ = rag(
        query=query,
        k=k,
        model_name=model_name,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        vectorstore=vectorstore
    )

    # Return only the answer
    return jsonify({
        "answer": generated_answer
    })


# ---------------------------------------------------------
# Run the Flask application
# ---------------------------------------------------------

if __name__ == "__main__":
    rag_api.run(debug=True)

The Flask app (`app.py`) is written to the `backend` folder. It defines three POST endpoints:

- `/v1/relevant_chunks` returns only the retrieved chunks for a given query.
- `/v1/answer_with_relevant_chunks` returns both the generated answer and the supporting chunks.
- `/v1/answer` returns only the generated answer.

The embedding model and FAISS index are loaded once at startup to avoid reloading them on every request. The `allow_dangerous_deserialization=True` flag is needed because FAISS indexes use pickle under the hood.

## Define the dependencies

The backend `requirements.txt` lists the Python packages needed to run the Flask API inside the Docker container.

In [ ]:
%%writefile backend/requirements.txt
# Code cell 042

flask==3.1.3
Werkzeug==3.1.8
requests==2.34.2
gunicorn==26.2.0
uvicorn[standard]
langchain-huggingface==1.2.2
langchain-community==0.4.2
langchain-openai==1.6.1
sentence-transformers==6.1.0
faiss-cpu==1.15.0

## Define the Dockerfile

- The Dockerfile defines how to build the backend container image.
- It installs all dependencies and pre-downloads the embedding model during the build step.

In [ ]:
%%writefile backend/Dockerfile
# Code cell 043

FROM python:3.13.15-slim

WORKDIR /app
COPY . .

RUN pip install --no-cache-dir --upgrade pip

RUN pip install --no-cache-dir \
    torch \
    --index-url https://download.pytorch.org/whl/cpu

RUN pip install --no-cache-dir -r requirements.txt

ENV HF_HOME=/app/huggingface

# Download the model at build time
RUN python -c "\
from langchain_huggingface import HuggingFaceEmbeddings; \
HuggingFaceEmbeddings( \
    model_name='HarishMaths/Hotel-Policy-Embedding', \
    model_kwargs={'device': 'cpu'}, \
    encode_kwargs={'normalize_embeddings': True} \
)"

# Force offline mode at runtime so it uses the cached model
ENV TRANSFORMERS_OFFLINE=1
ENV HF_HUB_OFFLINE=1

CMD ["gunicorn", "-w", "1", "--timeout", "120", "-b", "0.0.0.0:7860", "app:rag_api"]

A few important things about this Dockerfile:

- **PyTorch CPU-only build** is installed from a special index URL to avoid pulling in the much larger GPU version.
- The **embedding model is downloaded at build time** using a RUN command. This means the model is baked into the image.
- **TRANSFORMERS_OFFLINE=1** and **HF_HUB_OFFLINE=1** are set so the container never tries to reach Hugging Face at runtime, making startup faster and more reliable.
- **Gunicorn** runs the Flask app with one worker and a 120-second timeout on port 7860.

# Deployment - Frontend

## Define the Streamlit application

We create a separate `frontend` directory for the Streamlit UI components.

In [ ]:
# Code cell 044
# Create a folder for storing the files needed for frontend UI deployment
os.makedirs("frontend", exist_ok=True)

- The Streamlit app provides a simple web interface where hotel staff can type their questions and get answers from the RAG system.
- It communicates with the Flask backend through HTTP requests.

In [ ]:
%%writefile frontend/app.py
# Code cell 045

import streamlit as st
import requests


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

BACKEND_URL = "http://localhost:7860"


# ---------------------------------------------------------
# Page configuration
# ---------------------------------------------------------

st.set_page_config(
    page_title="Aurora Stays",
    page_icon="🏨",
    layout="wide"
)


# ---------------------------------------------------------
# Sidebar  -  Model parameters
# ---------------------------------------------------------

with st.sidebar:

    st.header("⚙️ Parameters")

    k = st.slider(
        "k (chunks to retrieve)",
        min_value=1,
        max_value=10,
        value=5,
        help="Number of relevant chunks fetched from the vector store."
    )

    temperature = st.slider(
        "Temperature",
        min_value=0.0,
        max_value=1.0,
        value=0.0,
        step=0.1,
        help="Higher values make the output more creative; lower values make it more focused."
    )

    max_tokens = st.slider(
        "Max tokens",
        min_value=64,
        max_value=2048,
        value=512,
        step=64,
        help="Maximum number of tokens in the generated answer."
    )


# ---------------------------------------------------------
# Main area
# ---------------------------------------------------------

st.title("🏨 Aurora Stays")
st.write("Ask a question about Aurora Stays policies.")


# ---------------------------------------------------------
# Response type selection (two options only)
# ---------------------------------------------------------

response_type = st.radio(
    "Response mode",
    ["Answer only", "Answer + Relevant Chunks"],
    horizontal=True
)


# ---------------------------------------------------------
# Query input
# ---------------------------------------------------------

query = st.text_input(
    "Your question",
    placeholder="e.g., What is the cancellation policy for a flexible reservation?"
)


# ---------------------------------------------------------
# Submit
# ---------------------------------------------------------

if st.button("Submit", type="primary"):

    if not query.strip():
        st.warning("Please enter a question.")

    else:

        # Pick endpoint
        if response_type == "Answer only":
            endpoint = "/v1/answer"
        else:
            endpoint = "/v1/answer_with_relevant_chunks"

        # Build request payload with parameters
        payload = {
            "query": query,
            "k": k,
            "temperature": temperature,
            "max_tokens": max_tokens
        }

        try:
            with st.spinner("Thinking…"):
                response = requests.post(
                    f"{BACKEND_URL}{endpoint}",
                    json=payload
                )

            if response.status_code == 200:

                result = response.json()

                # -- Answer only --------------------------------
                if response_type == "Answer only":

                    st.subheader("Answer")
                    st.write(result["answer"])

                # -- Answer + Relevant Chunks -------------------
                else:

                    st.subheader("Answer")
                    st.write(result["answer"])

                    st.subheader("Relevant Chunks")
                    for i, chunk in enumerate(result["relevant_chunks"], start=1):
                        with st.expander(f"Chunk {i}"):
                            st.write(chunk)

            else:
                st.error(
                    f"Unable to get a response from the RAG API. "
                    f"Status code: {response.status_code}"
                )

        except requests.exceptions.RequestException:
            st.error(
                "Unable to connect to the RAG API. "
                "Please make sure the backend service is running."
            )

The frontend app includes:

- A **sidebar** with adjustable parameters for k (number of chunks), temperature, and max tokens.
- A **radio button** to choose between "Answer only" and "Answer + Relevant Chunks" modes.
- A **text input** field for the query and a submit button.
- **Error handling** for both HTTP errors and connection failures.

When the user submits a question, the app sends a POST request to the appropriate backend endpoint and displays the response. In the "Answer + Relevant Chunks" mode, each supporting chunk is shown inside an expandable section.

## Define the dependencies

The frontend only needs two packages: `requests` for making HTTP calls to the backend, and `streamlit` for the UI framework.

In [ ]:
%%writefile frontend/requirements.txt
# Code cell 046

requests==2.34.2
streamlit==1.64.0

## Define the Dockerfile

The frontend Dockerfile is simpler than the backend one since it only needs to install Streamlit and run the app.

In [ ]:
%%writefile frontend/Dockerfile
# Code cell 047

# Use a minimal base image with Python 3.13.15-slim installed
FROM python:3.13.15-slim

# Set the working directory inside the container
WORKDIR /app

# Copy all files from the current directory to the container's /app directory
COPY . .

# Upgrade pip
RUN pip install --no-cache-dir --upgrade pip

# Install Python dependencies listed in requirements.txt
RUN pip install -r requirements.txt

# Run the Streamlit application on port 8501
# 0.0.0.0 makes the application accessible outside the container
# XSRF protection is disabled for easier external access
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

- The frontend container runs Streamlit on port 8501 and binds to `0.0.0.0` so it is accessible outside the container.
- XSRF protection is disabled to allow external access from the Codespace forwarded URL.
- The image is lightweight since it only needs Python, Streamlit, and the requests library.

# Pushing the files to GitHub

**Points to note before executing the below cell**

- Create a GitHub account and generate a Personal Access Token by following the instructions provided on the document titled **`Introduction to GitHub and GitHub Codespaces`** and **`Creating GitHub Account and Generating Personal Access Token`**.
- Once you have the token, paste it in the code cell below in place of `YOUR_GITHUB_TOKEN`.

## Local credentials setup

This notebook runs locally in Jupyter. Do not paste API keys or GitHub tokens into notebook cells. Copy `config/config.example.json` to `config/config.json`, fill in the values locally, and keep `config/config.json` out of version control.

Required credentials:

- `OPENAI_API_KEY` for the RAG and evaluation models.
- `OPENAI_BASE_URL` for the OpenAI-compatible endpoint.
- `GITHUB_TOKEN` for the optional repository upload section.

The GitHub upload section is optional. Run it only after configuring a token with the minimum repository permissions needed for your target repository.

Before running the GitHub section, verify that `config/config.json` is present locally and is not tracked by Git.

The remaining deployment instructions refer to the local `backend/` and `frontend/` folders generated by this notebook.

The notebook uses local files and environment/configuration values; no Google Colab integration is required.

In [ ]:
# Code cell 048
# Load the GitHub Personal Access Token from the local config file.
# Keep config/config.json private and never commit the token.
GITHUB_TOKEN = config.get("GITHUB_TOKEN") or os.getenv("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise ValueError("Add GITHUB_TOKEN to config/config.json or set it as an environment variable.")


The GitHub token is read from local configuration or the `GITHUB_TOKEN` environment variable and is never written into notebook source.

1. **Define authentication and repository details:**
   - `GITHUB_TOKEN` stores the GitHub Personal Access Token for authentication.
   - `REPO_NAME` specifies the name of the new GitHub repository.

2. **Authenticate with GitHub:**
   - `g = Github(GITHUB_TOKEN)` creates a GitHub client using the token.
   - `user = g.get_user()` retrieves the authenticated user.

3. **Create the repository:**
   - `user.create_repo()` creates a new public repository on GitHub with an auto-generated README.

4. **Push files to the repository:**
   - `repo.create_file()` uploads each deployment file to the repository.
   - Backend files (`app.py`, `requirements.txt`, `Dockerfile`, model) go into the `backend/` folder.
   - Frontend files (`app.py`, `requirements.txt`, `Dockerfile`) go into the `frontend/` folder.
   - For the `.joblib` model file, which is binary, PyGithub handles the encoding automatically.

In [ ]:
# Code cell 049
# Name for the repository
REPO_NAME = "aurora-stay-qna"

# Authenticate with GitHub
from github import Github
import os

g = Github(GITHUB_TOKEN)
user = g.get_user()

print(f"Authenticated as: {user.login}")


# ---------------------------------------------------------
# Get existing repository or create a new one
# ---------------------------------------------------------

try:
    # Check if repository already exists
    repo = user.get_repo(REPO_NAME)
    print(f"Repository already exists: {repo.html_url}")

except Exception:
    # Repository does not exist, so create it
    repo = user.create_repo(
        REPO_NAME,
        description="Aurora RAG",
        private=False,
        auto_init=True
    )

    print(f"Repository created successfully: {repo.html_url}")


# ---------------------------------------------------------
# Files to push
# ---------------------------------------------------------

files_to_push = {
    "backend/app.py": "backend/app.py",
    "backend/rag_utils.py": "backend/rag_utils.py",
    "backend/requirements.txt": "backend/requirements.txt",
    "backend/Dockerfile": "backend/Dockerfile",

    "frontend/app.py": "frontend/app.py",
    "frontend/requirements.txt": "frontend/requirements.txt",
    "frontend/Dockerfile": "frontend/Dockerfile",
}


# ---------------------------------------------------------
# Function to upload or update a file
# ---------------------------------------------------------

def upload_or_update_file(repo, repo_path, local_path):

    with open(local_path, "rb") as f:
        content = f.read()

    try:
        # Check if the file already exists
        existing_file = repo.get_contents(
            repo_path,
            ref="main"
        )

        # File exists -> update it
        repo.update_file(
            path=repo_path,
            message=f"Update {repo_path}",
            content=content,
            sha=existing_file.sha,
            branch="main"
        )

        print(f"Updated: {repo_path}")

    except Exception:
        # File does not exist -> create it
        repo.create_file(
            path=repo_path,
            message=f"Add {repo_path}",
            content=content,
            branch="main"
        )

        print(f"Uploaded: {repo_path}")


# ---------------------------------------------------------
# Upload / update individual files
# ---------------------------------------------------------

for repo_path, local_path in files_to_push.items():

    upload_or_update_file(
        repo,
        repo_path,
        local_path
    )

# ---------------------------------------------------------
# Upload / update FAISS index folder
# ---------------------------------------------------------

faiss_folder = "backend/faiss_index"

if os.path.exists(faiss_folder):

    for root, dirs, files in os.walk(faiss_folder):

        for file_name in files:

            # Full local path
            local_path = os.path.join(
                root,
                file_name
            )

            # Path relative to backend/
            relative_path = os.path.relpath(
                local_path,
                "backend"
            )

            # GitHub repository path
            repo_path = os.path.join(
                "backend",
                relative_path
            ).replace("\\", "/")

            upload_or_update_file(
                repo,
                repo_path,
                local_path
            )

else:
    print(
        f"Warning: FAISS folder not found: {faiss_folder}"
    )


print("\nAll files pushed successfully!")
print(f"Repository: {repo.html_url}")

Once all the files have been pushed, the next step is to add the secrets (OpenAI API Key and OpenAI Base URL) to GitHub Codespaces so that the backend can connect to OpenAI. For this, refer to the document titled **"Adding Secrets to GitHub Codespaces."**

**Note:** The link to your GitHub repository will be displayed as the output of the previous cell.

Then, refer to the document titled **"Deploying Frontend and Backend on GitHub Codespaces"** for the next steps to deploy the Streamlit application and see it in action.

# Deployment - Inference

Before executing the code cells below, ensure the following:

1. GitHub Codespaces is up and running.

2. Both the frontend and backend applications are running in GitHub Codespaces, and the ports are set to **Public**.

3. The forwarded URL of the backend is copied.(More details are provided in the document titled **"Deploying Frontend and Backend on GitHub Codespaces"**)

In [ ]:
# Code cell 050
# Set the base URL for the deployed Flask API running in GitHub Codespaces
model_root_url = "_____"  # Replace with your Codespace forwarded URL for port 7860

Replace the URL above with the actual forwarded URL from your Codespace's Ports tab. It will look something like `https://organic-space-abcd1234-7860.app.github.dev`. Make sure there is no trailing slash at the end.

This URL is a tunnel that Codespaces creates to route external HTTP requests into your Codespace and to the Flask container running on port 7860.

In [ ]:
# Code cell 051
model_chunks_url = model_root_url + "/v1/relevant_chunks"  # Endpoint for getting relevant chunks

Since our model predictions are served through the Flask endpoint we created, we need to call this endpoint to get the relevant chunks.

> ```@app.post('/v1/relevant_chunks')```

In [ ]:
# Code cell 052
model_ans_chunks_url = model_root_url + "/v1/answer_with_relevant_chunks"  # Endpoint for getting the answer with relevant chunks

> ```@app.post('/v1/answer_with_relevant_chunks')```

The idea is to send a single request to the API and receive an immediate response. This is useful for real-time applications like recommendation systems and fraud detection.

* This data is sent as a JSON payload in a POST request to the model endpoint.
* The model processes the input features and returns a prediction as a JSON payload.

## Deployment - Inference (Relevant chunks)

**Guidelines**

- Enter a **hotel policy related question** in `query` to test the backend application.
- Set `k` to the number of policy chunks to retrieve. A value around **2 to 5** is a good starting point.
- You can try questions about **cancellations, refunds, pet policies, additional charges, room information, or accessibility**.
- Try different questions and `k` values to understand how they affect the retrieved information and the final response.

In [ ]:
# Code cell 053
# Define the test payload with a sample hotel policy question and the number of chunks to retrieve
payload = {
    "query": "_____",  # Enter a hotel policy question to test the backend
    "k": _____,        # Enter the number of chunks to retrieve
}

In [ ]:
# Code cell 054
# Sending a POST request to the model endpoint with the test payload
response = requests.post(model_chunks_url, json=payload)

In [ ]:
# Code cell 055
# Check the HTTP status of the response
response

- The `<Response [200]>` you see is an HTTP status code.
- It indicates that your request was successful, and the server was able to process it without any problems.

In [ ]:
# Code cell 056
# Print the full JSON response containing the retrieved chunks
print(response.json())

- The JSON output contains the original query, the value of k, and a list of the top-k retrieved chunks.
- Each chunk includes the text content and metadata showing which source document it came from.
- This confirms the retrieval endpoint is working correctly on the deployed backend.

## Deployment - Inference (Answer with relevant chunks)

**Guidelines**

- Enter a **hotel policy related question** in `query` to test the answer endpoint.
- Set `k` to the number of policy chunks to retrieve. A value around **2 to 5** is a good starting point.
- The `model_name` specifies the LLM used to generate the answer.
- Set `temperature` to a low value, such as **0 to 0.2**, for more consistent responses.
- Set `top_p` around **0.9 to 1** for consistent token selection.
- Set `max_tokens` based on the expected answer length. **256 to 512** is a reasonable starting range.
- The values can be adjusted to understand how the retrieval and generation settings affect the response.

In [ ]:
# Code cell 057
# Define the test payload with the query and model parameters for the answer endpoint
payload = {
    "query": "_____",          # Enter a hotel policy question to test the answer endpoint
    "k": _____,                 # Enter the number of chunks to retrieve
    "model_name": "gpt-4o-mini", # Enter the LLM model name
    "temperature": _____,       # Enter a low value for more consistent responses
    "top_p": _____,             # Enter a value close to 1 for balanced token selection
    "max_tokens": _____         # Enter a suitable limit based on the expected answer length
}

**Note:** The below code snippet may take some time to execute, typically between 1 to 3 minutes.

In [ ]:
# Code cell 058
# Sending a POST request to the model endpoint with the test payload
response = requests.post(model_ans_chunks_url, json=payload)

In [ ]:
# Code cell 059
# Check the HTTP status of the response
response

- The `<Response [200]>` you see is an HTTP status code.
- It indicates that your request was successful, and the server was able to process it without any problems.

In [ ]:
# Code cell 060
# Print the full JSON response containing the answer and supporting chunks
print(response.json())

- The JSON response now includes both the generated answer and the relevant chunks that were used to produce it.
- This is the same payload the Streamlit frontend receives when the user selects "Answer + Relevant Chunks" mode.
- Having both the answer and its supporting evidence in a single response makes it easy for hotel staff to verify the information before sharing it with guests.

**IMPORTANT:** Once the previous cell has been executed and the application is no longer required, please refer to the document titled **"Deleting Codespace"** to delete the Codespace, as it may continue to consume resources.

# Business Insights and Recommendations

## Evaluation findings

The K=5 baseline uses chunk_size=512 and chunk_overlap=64. On the dev set (evaluation.csv), the system achieved a 60% overall pass rate: Faithfulness 0.96, Contextual Recall 0.79, Contextual Precision 0.60, and Answer Relevancy 0.72.

When the dev pipeline was evaluated with the same test.xlsx file used for the held-out production/test evaluation, the results matched: 30% overall pass rate, Faithfulness 0.97, Contextual Recall 0.85, Contextual Precision 0.62, and Answer Relevancy 0.86. Therefore, the earlier dev-versus-prod difference was caused by different datasets, not different models.

### Per-test-case metrics — test.xlsx

**Test summary:** 3 of 10 test cases passed and 7 failed (30% overall pass rate).

| Test outcome | Cases | Rate |
|---|---:|---:|
| Passed | 3 | 30% |
| Failed | 7 | 70% |
| **Total** | **10** | **100%** |

Scores are rounded to three decimals; case numbers follow the order in `test.xlsx`.

| Case | Question summary | Faithfulness | Recall | Precision | Relevancy | Composite | Route |
|---:|---|---:|---:|---:|---:|---:|---|
| 1 | OTA cancellation/refund responsibility | 0.906 | 0.856 | 0.786 | 0.940 | 0.876 | Auto answer |
| 2 | Member discount and free cancellation | 1.000 | 0.969 | 0.647 | 0.966 | 0.915 | Auto answer |
| 3 | Double Queen occupancy | 1.000 | 0.874 | 0.625 | 0.997 | 0.893 | Auto answer |
| 4 | Accessible King room inclusions | 0.995 | 0.982 | 0.371 | 0.748 | 0.818 | Auto answer |
| 5 | Atlanta and Miami parking fees | 0.973 | 0.920 | 0.687 | 0.943 | 0.897 | Auto answer |
| 6 | Miami Brickell breakfast price | 1.000 | 0.991 | 0.627 | 0.993 | 0.922 | Auto answer |
| 7 | Advance Purchase date change | 0.926 | 0.868 | 0.642 | 0.905 | 0.851 | Auto answer |
| 8 | Denver late checkout at 2 PM | 0.926 | 0.185 | 0.343 | 0.250 | 0.489 | Manual lookup |
| 9 | ASH-CHI Loop non-service pet fee | 1.000 | 0.916 | 0.792 | 1.000 | 0.937 | Auto answer |
| 10 | Incidental hold release timing | 0.998 | 0.970 | 0.783 | 0.903 | 0.929 | Auto answer |

**Test summary:** 10 test cases total — **3 passed**, **7 failed** (**30% pass rate**). The routing column is a separate operational measure: 9 cases were routed to Auto Answer and 1 to Manual Lookup.

**Overall averages across all 10 test cases:** Faithfulness **0.972**, Contextual Recall **0.853**, Contextual Precision **0.630**, Answer Relevancy **0.864**, and Composite Score **0.853**.

Chunk-size testing with the existing architecture and data showed that 512 tokens with 64-token overlap outperformed 400/64 and 450/80 on both dev and test retrieval coverage.

## Recommendations for contextual precision

- Keep the baseline retrieval setting at k=5, chunk_size=512, and chunk_overlap=64.
- Improve chunk boundaries so policy sections and property-matrix rows remain intact.
- For future iterations, test a retrieve-more-then-select approach: retrieve candidate chunks, rank them for direct relevance, and pass only the best 3-5 chunks to the generator.
- Preserve exact property names, amounts, and times in chunks because these are frequent sources of retrieval noise.
- Continue reporting Contextual Precision separately; it is the current bottleneck even when Faithfulness and Answer Relevancy are strong.
- Compare dev and production using the same dataset before attributing score differences to model or deployment changes.